# Weekly WTI Multivariate Log-Return H2 Fold48

- Dataset: weekly `data/0428DB_weekly.csv`
- Target: `Com_CrudeOil`
- Horizon: 2 weeks
- Folds: 48 rolling windows, step 1
- Transform: target only log-return; exogenous variables stay in level scale
- Models: GRU, TimeXer, iTransformer
- Outputs: MSE/MAE/MAPE metrics, fold forecast path graph, actual-vs-predicted graph, loss graphs

Note: WTI has a historical non-positive weekly price. For strict log-return math, the runner starts after the last non-positive target date and records this in `run_config.json`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import importlib
import importlib.util
import os
import subprocess
import sys
from pathlib import Path

BOOTSTRAP_SENTINEL = Path('/content/newoil_colab_bootstrap_weekly_logret_h2_v1')

# Do not reinstall torch in Colab. Reinstalling torch is what caused the previous
# torch/torchvision CUDA-version mismatch. We keep Colab's torch and remove optional
# vision/audio packages that torchmetrics may import indirectly.
PACKAGE_SPECS = [
    'numpy>=1.26,<2.2',
    'pandas==2.2.2',
    'protobuf>=4.25,<6',
    'tensorboard>=2.18,<2.20',
    'pyyaml==6.0.2',
    'matplotlib>=3.8,<3.11',
    'rich>=13,<15',
    'utilsforecast',
    'coreforecast',
    'lightning-utilities>=0.11,<0.16',
    'torchmetrics>=1.6,<1.9',
    'pytorch-lightning>=2.4,<2.6',
    'ray[tune]>=2.20,<3.0',
]
NEURALFORECAST_SPEC = 'neuralforecast==3.1.7'
OPTIONAL_TORCH_PACKAGES_TO_REMOVE = ['torchvision', 'torchaudio']
CRITICAL_IMPORTS = [
    'numpy',
    'pandas',
    'matplotlib',
    'yaml',
    'torch',
    'pytorch_lightning',
    'torchmetrics',
    'ray',
    'neuralforecast',
]


def import_is_healthy(module_name):
    try:
        importlib.import_module(module_name)
        return True
    except Exception as exc:
        print(f'Import check failed for {module_name}: {type(exc).__name__}: {exc}')
        return False


installed_optional_torch_packages = [
    package
    for package in OPTIONAL_TORCH_PACKAGES_TO_REMOVE
    if importlib.util.find_spec(package) is not None
]

bootstrap_changed = False

if installed_optional_torch_packages:
    subprocess.run(
        [sys.executable, '-m', 'pip', 'uninstall', '-y', *installed_optional_torch_packages],
        check=True,
    )
    bootstrap_changed = True

needs_repair = (not BOOTSTRAP_SENTINEL.exists()) or any(
    not import_is_healthy(module_name) for module_name in CRITICAL_IMPORTS
)

if needs_repair:
    subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '-q', '--no-cache-dir', '--upgrade', *PACKAGE_SPECS],
        check=True,
    )
    subprocess.run(
        [
            sys.executable,
            '-m',
            'pip',
            'install',
            '-q',
            '--no-cache-dir',
            '--no-deps',
            '--upgrade',
            NEURALFORECAST_SPEC,
        ],
        check=True,
    )
    bootstrap_changed = True

if bootstrap_changed:
    BOOTSTRAP_SENTINEL.write_text('ready\n', encoding='utf-8')
    print('Bootstrap installed/repaired packages. Runtime will restart once. Rerun this cell after restart.')
    os.kill(os.getpid(), 9)

sanity_code = 'import numpy, pandas, torch, pytorch_lightning, torchmetrics, ray, neuralforecast; print("import sanity ok")'
sanity = subprocess.run([sys.executable, '-c', sanity_code], capture_output=True, text=True)
if sanity.returncode != 0:
    print(sanity.stdout)
    print(sanity.stderr)
    raise RuntimeError('Colab package sanity check failed. Restart runtime and rerun this cell.')

print(sanity.stdout.strip())

In [ ]:
from pathlib import Path
import subprocess, sys, os

WORKDIR = Path('/content/newoil')
REPO_URL = 'https://github.com/Jaeho777/newoil.git'

if WORKDIR.exists():
    subprocess.run(['git', '-C', str(WORKDIR), 'fetch', 'origin', 'main'], check=True)
    subprocess.run(['git', '-C', str(WORKDIR), 'reset', '--hard', 'origin/main'], check=True)
else:
    subprocess.run(['git', 'clone', REPO_URL, str(WORKDIR)], check=True)

commit = subprocess.check_output(['git', '-C', str(WORKDIR), 'rev-parse', '--short', 'HEAD'], text=True).strip()
print('Using newoil commit:', commit)

In [ ]:
VARIABLES = [
    'USEPUINDXD',
    'Idx_SnPVIX',
    'Com_DubaiOil',
    'Com_OmanOil',
    'Com_BrentCrudeOil',
    'Com_CrudeOil',
    'Com_Petronet_Kerosene',
    'Com_Petronet_HSFO_180cst_3_5pct',
    'Com_Petronet_Naphtha',
    'Com_Petronet_Gasoline_92RON',
    'GPRD_THREAT',
    'GPRD',
    'GPRD_ACT',
    'N10D',
]
print('Variables:', len(VARIABLES))
print(VARIABLES)

In [ ]:
OUTPUT_ROOT = Path('/content/drive/MyDrive/newoil_outputs')
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
commit = subprocess.check_output(['git', '-C', str(WORKDIR), 'rev-parse', '--short', 'HEAD'], text=True).strip()
print('Running newoil commit:', commit, flush=True)

cmd = [
    sys.executable, '-u',
    str(WORKDIR / 'scripts' / 'run_weekly_logret_h2_fold48.py'),
    '--repo-root', str(WORKDIR),
    '--output-root', str(OUTPUT_ROOT),
    '--horizon', '2',
    '--n-windows', '48',
    '--step-size', '1',
    '--val-size', '48',
    '--input-size', '64',
    '--max-epochs', '500',
]
print('RUN:', ' '.join(cmd), flush=True)
subprocess.run(cmd, check=True)

In [ ]:
import pandas as pd
from IPython.display import Image, display

runs = sorted(OUTPUT_ROOT.glob('weekly_logret_h2_fold48_*'), key=lambda p: p.stat().st_mtime)
latest = runs[-1]
print('Latest output:', latest)
run_config = pd.read_json(latest / 'run_config.json', typ='series')
loss_overview = latest / 'loss_curves_overview.png'
if not loss_overview.exists():
    raise FileNotFoundError(f'{loss_overview} not found. This is an old run; rerun all cells after pulling the latest commit.')
print('Run config commit/status check:')
config_keys = ['target_transform', 'exog_transform', 'neuralforecast_scaler_type', 'optimizer', 'weight_decay']
display(run_config[[key for key in config_keys if key in run_config.index]])

summary = pd.read_csv(latest / 'fold48_metrics_summary.csv')
display(summary)
loss_summary = pd.read_csv(latest / 'loss_summary.csv')
display(loss_summary)
leakage_audit = pd.read_csv(latest / 'leakage_audit.csv')
display(leakage_audit)
if not leakage_audit['passed'].all():
    raise RuntimeError('Leakage audit failed. Inspect leakage_audit.csv before using results.')

display(Image(filename=str(latest / 'fold_forecast_paths.png')))
display(Image(filename=str(latest / 'actual_vs_predicted_price.png')))
display(Image(filename=str(loss_overview)))